In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, to_date, to_timestamp, datediff, coalesce, lit,trim,expr
from pyspark.sql.types import StringType, IntegerType, TimestampType, IntegerType, DateType


In [0]:
df = spark.table("workspace.bronze.reviews")

In [0]:
trim_cols = ["order_id", "review_id", "review_comment_title", "review_comment_message"]
for c in trim_cols:
    df = df.withColumn(c, trim(col(c)))

df = df.withColumn("review_comment_message", 
                   F.regexp_replace(col("review_comment_message"), r"[\n\r]", " "))

In [0]:
df = df.withColumn("review_score", F.expr("try_cast(review_score as int)"))
df = df.withColumn("review_score", F.coalesce(F.col("review_score"), F.lit(0)))


df = df.withColumn("review_creation_date", F.to_timestamp(F.col("review_creation_date")))
df = df.withColumn("review_answer_timestamp", F.to_timestamp(F.col("review_answer_timestamp")))


In [0]:
df = df.filter(col("review_id").isNotNull() & col("order_id").isNotNull())
df = df.dropDuplicates(["review_id", "order_id"])

In [0]:
df = df.withColumn("response_lag_days", 
                   F.coalesce(
                       F.datediff(F.col("review_answer_timestamp"), F.col("review_creation_date")), 
                       F.lit(-1)
                   ).cast(IntegerType()) 
                  )

In [0]:
df = df.withColumn("silver_processed_time", F.current_timestamp())


In [0]:
df.display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.order_reviews")